In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, desc, round

In [4]:
spark = SparkSession.builder.appName("Student Exam Performance Analysis").getOrCreate()

In [5]:
df = spark.read.csv("StudentsPerformance.csv", header=True, inferSchema=True)

In [6]:
print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))

Total Rows: 1000
Total Columns: 8


In [7]:
print("Dataset Preview:")
df.show(5)

Dataset Preview:
+------+--------------+---------------------------+------------+-----------------------+----------+-------------+-------------+
|gender|race/ethnicity|parental level of education|       lunch|test preparation course|math score|reading score|writing score|
+------+--------------+---------------------------+------------+-----------------------+----------+-------------+-------------+
|female|       group B|          bachelor's degree|    standard|                   none|        72|           72|           74|
|female|       group C|               some college|    standard|              completed|        69|           90|           88|
|female|       group B|            master's degree|    standard|                   none|        90|           95|           93|
|  male|       group A|         associate's degree|free/reduced|                   none|        47|           57|           44|
|  male|       group C|               some college|    standard|                   none

In [8]:
print("Schema:")
df.printSchema()

Schema:
root
 |-- gender: string (nullable = true)
 |-- race/ethnicity: string (nullable = true)
 |-- parental level of education: string (nullable = true)
 |-- lunch: string (nullable = true)
 |-- test preparation course: string (nullable = true)
 |-- math score: integer (nullable = true)
 |-- reading score: integer (nullable = true)
 |-- writing score: integer (nullable = true)



In [9]:
print("Missing Values Check:")
for column in df.columns:
    print(column, df.filter(col(column).isNull()).count())


Missing Values Check:
gender 0
race/ethnicity 0
parental level of education 0
lunch 0
test preparation course 0
math score 0
reading score 0
writing score 0


In [10]:
clean_df = df.dropna()

In [11]:
analysis_df = clean_df.withColumn("total_score", col("math score") + col("reading score") + col("writing score")).withColumn("average_score",round(col("total_score") / 3, 2))

In [12]:
print("Subject-wise Average Scores:")

subject_avg = analysis_df.select(
    round(avg("math score"), 2).alias("avg_math_score"),
    round(avg("reading score"), 2).alias("avg_reading_score"),
    round(avg("writing score"), 2).alias("avg_writing_score"),
    round(avg("average_score"), 2).alias("overall_avg_score")
)

subject_avg.show()

Subject-wise Average Scores:
+--------------+-----------------+-----------------+-----------------+
|avg_math_score|avg_reading_score|avg_writing_score|overall_avg_score|
+--------------+-----------------+-----------------+-----------------+
|         66.09|            69.17|            68.05|            67.77|
+--------------+-----------------+-----------------+-----------------+



In [13]:
print("Dataset with Total and Average Score:")
analysis_df.show(5)

Dataset with Total and Average Score:
+------+--------------+---------------------------+------------+-----------------------+----------+-------------+-------------+-----------+-------------+
|gender|race/ethnicity|parental level of education|       lunch|test preparation course|math score|reading score|writing score|total_score|average_score|
+------+--------------+---------------------------+------------+-----------------------+----------+-------------+-------------+-----------+-------------+
|female|       group B|          bachelor's degree|    standard|                   none|        72|           72|           74|        218|        72.67|
|female|       group C|               some college|    standard|              completed|        69|           90|           88|        247|        82.33|
|female|       group B|            master's degree|    standard|                   none|        90|           95|           93|        278|        92.67|
|  male|       group A|         associ

In [14]:
print("Top 5 Students by Average Score:")
top_5 = analysis_df.orderBy(desc("average_score")).limit(5)
top_5.show()

Top 5 Students by Average Score:
+------+--------------+---------------------------+--------+-----------------------+----------+-------------+-------------+-----------+-------------+
|gender|race/ethnicity|parental level of education|   lunch|test preparation course|math score|reading score|writing score|total_score|average_score|
+------+--------------+---------------------------+--------+-----------------------+----------+-------------+-------------+-----------+-------------+
|female|       group E|          bachelor's degree|standard|                   none|       100|          100|          100|        300|        100.0|
|  male|       group E|          bachelor's degree|standard|              completed|       100|          100|          100|        300|        100.0|
|female|       group E|         associate's degree|standard|                   none|       100|          100|          100|        300|        100.0|
|female|       group E|          bachelor's degree|standard|       

In [15]:
print("Gender-wise Average Score:")
gender_avg = analysis_df.groupBy("gender") \
    .agg(round(avg("average_score"), 2).alias("avg_score")) \
    .orderBy(desc("avg_score"))
gender_avg.show()

Gender-wise Average Score:
+------+---------+
|gender|avg_score|
+------+---------+
|female|    69.57|
|  male|    65.84|
+------+---------+



In [16]:
print("Parental Education-wise Average Score:")
parent_avg = analysis_df.groupBy("parental level of education") \
    .agg(round(avg("average_score"), 2).alias("avg_score")) \
    .orderBy(desc("avg_score"))
parent_avg.show()

Parental Education-wise Average Score:
+---------------------------+---------+
|parental level of education|avg_score|
+---------------------------+---------+
|            master's degree|     73.6|
|          bachelor's degree|    71.92|
|         associate's degree|    69.57|
|               some college|    68.48|
|           some high school|    65.11|
|                high school|     63.1|
+---------------------------+---------+



In [17]:
print("Test Preparation Course Impact:")
prep_avg = analysis_df.groupBy("test preparation course") \
    .agg(round(avg("average_score"), 2).alias("avg_score")) \
    .orderBy(desc("avg_score"))
prep_avg.show()

Test Preparation Course Impact:
+-----------------------+---------+
|test preparation course|avg_score|
+-----------------------+---------+
|              completed|    72.67|
|                   none|    65.04|
+-----------------------+---------+



In [18]:
print("Lunch Type Impact:")
lunch_avg = analysis_df.groupBy("lunch") \
    .agg(round(avg("average_score"), 2).alias("avg_score")) \
    .orderBy(desc("avg_score"))
lunch_avg.show()

Lunch Type Impact:
+------------+---------+
|       lunch|avg_score|
+------------+---------+
|    standard|    70.84|
|free/reduced|     62.2|
+------------+---------+



In [19]:
gender_avg.show()
prep_avg.show()
lunch_avg.show()
parent_avg.show()

+------+---------+
|gender|avg_score|
+------+---------+
|female|    69.57|
|  male|    65.84|
+------+---------+

+-----------------------+---------+
|test preparation course|avg_score|
+-----------------------+---------+
|              completed|    72.67|
|                   none|    65.04|
+-----------------------+---------+

+------------+---------+
|       lunch|avg_score|
+------------+---------+
|    standard|    70.84|
|free/reduced|     62.2|
+------------+---------+

+---------------------------+---------+
|parental level of education|avg_score|
+---------------------------+---------+
|            master's degree|     73.6|
|          bachelor's degree|    71.92|
|         associate's degree|    69.57|
|               some college|    68.48|
|           some high school|    65.11|
|                high school|     63.1|
+---------------------------+---------+



In [20]:
import matplotlib.pyplot as plt

In [ ]:
gender_pd = gender_avg.toPandas()

plt.figure(figsize=(6,4))
plt.bar(gender_pd["gender"], gender_pd["avg_score"])
plt.xlabel("Gender")
plt.ylabel("Average Score")
plt.title("Gender-wise Average Score")
plt.show()

In [ ]:
parent_pd = parent_avg.toPandas()

plt.figure(figsize=(10,5))
plt.bar(parent_pd["parental level of education"], parent_pd["avg_score"])
plt.xlabel("Parental Level of Education")
plt.ylabel("Average Score")
plt.title("Parental Education-wise Average Score")
plt.xticks(rotation=45)
plt.show()

In [ ]:
prep_pd = prep_avg.toPandas()

plt.figure(figsize=(7,4))
plt.bar(prep_pd["test preparation course"], prep_pd["avg_score"])
plt.xlabel("Test Preparation Course")
plt.ylabel("Average Score")
plt.title("Impact of Test Preparation Course on Scores")
plt.show()

In [ ]:
try:
    spark.stop()
    print("Spark session stopped successfully.")
except NameError:
    print("Spark session was already stopped or not available.")